In [ ]:
# Step 1: Import Libraries
# Data Manipulation
# import StandardScaler
import pandas as pd
import numpy as np
import xgboost as xgb

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler,StandardScaler,RobustScaler

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    root_mean_squared_error
)

import warnings
warnings.filterwarnings('ignore')

# Step 2: Load Dataset
df = pd.read_csv("insurance (1).csv")
df.head()
# Why?
# Reads insurance dataset into DataFrame.
# Provides initial understanding of features and target variable.

# Step 3: Basic Data Exploration
df.shape
df.info()
df.describe()
df.isnull().sum()
df.duplicated().sum()
# Why?
# Checks:
# Number of rows and columns
# Data types
# Missing values
# Duplicate records
# Statistical summary

# Step 4: Exploratory Data Analysis (EDA)
# 4.1 Target Variable Distribution
plt.figure(figsize=(8,5))
sns.histplot(df["expenses"], kde=True)
plt.title("Insurance expenses Distribution")
plt.show()

# Why?
# Shows whether target variable is normally distributed.
# Expected Finding:
# Highly right-skewed
# This is why we later apply Log Transformation.

# 4.2 Age Distribution
plt.figure(figsize=(8,5))
sns.histplot(df["age"], bins=20)
plt.title("Age Distribution")
plt.show()
# 4.3 BMI Distribution
plt.figure(figsize=(8,5))
sns.histplot(df["bmi"], kde=True)
plt.title("BMI Distribution")
plt.show()
# 4.4 Smoker vs expenses
plt.figure(figsize=(8,5))
sns.boxplot(x="smoker", y="expenses", data=df)
plt.show()
# Why?
# Identifies impact of smoking on insurance premium.
# Key Insight:
# Smokers usually have significantly higher expenses.
# 4.5 Gender vs expenses
plt.figure(figsize=(8,5))
sns.boxplot(x="sex", y="expenses", data=df)
plt.show()
# 4.6 Region Analysis
plt.figure(figsize=(8,5))
sns.boxplot(x="region", y="expenses", data=df)
plt.show()
# 4.7 Correlation Heatmap
# First encode categorical columns.

eda_df = pd.get_dummies(df, drop_first=True)

plt.figure(figsize=(10,6))
sns.heatmap(
    eda_df.corr(),
    annot=True,
    cmap='coolwarm'
)
plt.show()
# Why?
# Identifies:
# Highly correlated features
# Strong predictors of expenses
# Expected Findings:
# Smoker has highest correlation with expenses.
# Age has positive correlation.

# Step 5: Outlier Analysis
plt.figure(figsize=(8,5))
sns.boxplot(y=df["expenses"])
plt.show()
# Why?
# Insurance expenses generally contain genuine high-value customers.
# In this dataset:
# ✅ Keep outliers
# ❌ Do not remove blindly
# Step 6: Feature Engineering
# Encoding Categorical Variables
df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

df_encoded.head()
# Why?
# Machine Learning algorithms only understand numbers.
# Converts:male/female,yes/no,regions,into numerical format.
# Step 7: Feature and Target Split
X = df_encoded.drop("expenses", axis=1)
y = df_encoded["expenses"]
# Step 8: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
# Why?
# Train on 80%
# Test on 20%
# Prevents data leakage.

# Step 9: Check Target Skewness
y_train.skew()
# Expected:> 1
# Indicates strong positive skewness.

# Step 10: Log Transformation
y_train_log = np.log(y_train)
y_test_log = np.log(y_test)
# Verify:
sns.histplot(y_train_log, kde=True)
plt.show()
# Why Log Transformation?
# Insurance expenses are heavily skewed.
# Problems without log transform:
# Large premiums dominate training.
# Linear models perform poorly.
# Higher prediction errors.
# Benefits:
# ✅ Reduces skewness
# ✅ Stabilizes variance
# ✅ Improves R²
# ✅ Improves linear relationship

# Step 11: Feature Scaling
# class StandardScaler:
#     pass
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Why Scaling?
# Algorithms sensitive to magnitude:
# Algorithm	Scaling Required
# Linear Regression	Yes
# Ridge	Yes
# Lasso	Yes
# ElasticNet	Yes
# KNN	Yes
# SVM	Yes
# Random Forest	No
# Decision Tree	No
# XGBoost	No
# For consistent comparison, scale all features.

# Step 12: Model Building
models = {
    "Linear Regression":LinearRegression(),
    "Ridge":Ridge(),
    "Lasso":Lasso(),
    "ElasticNet":ElasticNet(),
    "Decision Tree":DecisionTreeRegressor(),
    "Random Forest":RandomForestRegressor(),
    "Gradient Boosting":GradientBoostingRegressor(),
    # "XGBoost":xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
            }

# Step 13: Train and Evaluate Models
results = {}
for name, model in models.items():
    model.fit(X_train_scaled,y_train_log)
    y_pred_log = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test_log,y_pred_log)
    mse = mean_squared_error(y_test_log,y_pred_log)
    rmse = np.sqrt(mse)
    r2 = r2_score(
        y_test_log,
        y_pred_log
    )
    results[name] = {
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse
    }
# Step 14: Compare Models
results_df = pd.DataFrame(results).T
results_df.sort_values(
    by="R2",
    ascending=False
)

# Step 15: Visual Comparison of Models
results_df = results_df.sort_values(
    by='R2',
    ascending=False
)
plt.figure(figsize=(10,5))
sns.barplot(
    x=results_df.index,
    y=results_df["R2"]
)
plt.xticks(rotation=45)
plt.title(
    "Model Comparison by R² Score"
)
plt.show()

# Step 16: Final Prediction
best_model = GradientBoostingRegressor()
best_model.fit(
    X_train_scaled,
    y_train_log
)
pred_log = best_model.predict(
    X_test_scaled
)
pred_actual = np.exp(pred_log)
# Why Exponential?
# Since:y = log(expenses),To return to actual insurance amount:expenses = exp(y)